# Exercise 2: Tethys-Chloris (T&C): model setup, validation and sensitivity analysis

## Material
* Case Study Materials
  * Script to create the model inputs

* Input Data
  * Files containing CO2 air concentration

* Visualization Tools
  * Functions for results visualizations

## Objectives

This session is designed to help students learn how to set up a case study using the plot-scale version of the T&C model (Fatichi et al., 2012). The document outlines the key steps, including verifying critical model settings in new cases, understanding model inputs and outputs (especially in relation to vegetation dynamics), and conducting model validation. 

The tutorial here will focus on two case studies from Switzerland (Chamau and Laegeren).

# Part A: input preparation, model setup and validation

## 1. Load the Julia environment and set up file paths

Exercise 2 uses the same Julia environment configured in Exercise 1. Run the following cell first to activate this environment and define the paths to the CH-Lae input and output files.

In [ ]:
import Pkg

# Reuse the repository environment configured in Exercise 1.
Pkg.activate(joinpath(@__DIR__, ".."))

# Define all CH-Lae input and output paths relative to this notebook.
data_dir = joinpath(@__DIR__, "data")
forcing_input = joinpath(data_dir, "Data_CH-Lae_run.mat")
ca_path = joinpath(data_dir, "Ca_Data.mat")
yaml_path = joinpath(data_dir, "CH-Lae.yaml")
netcdf_path = joinpath(data_dir, "CH-Lae.nc")

## 2. Understand the case study

We begin with the case study in Chamau. Before running the simulation, answer the following questions:

- What are the mean annual precipitation and mean temperature in Chamau? (Refer to the data-access instructions in Exercise 1.)
- What is the vegetation type and land cover in Chamau? Is there any land management?
- What temporal patterns do you expect in Leaf Area Index (LAI) and Gross Primary Productivity (GPP)?
- What are the soil depth and soil texture? How many soil layers are set?

**Note:** In T&C, `ZR95 = 0` indicates that the corresponding vegetation type is absent, regardless of whether values have been assigned to its other parameters.

There are some hints in the previous exercise handout (exercise 1, part 3) about vegetation cover and land management. Remember to perform the same checks (understand the rainfall amount, vegetation type, land management, and soil depth and type) before running the simulation for a new case study. For example, try answering the same questions for Laegeren. While for the case study in Chamau all parameters were already provided, some key plant parameters and soil properties are not directly provided for the Laegeren case study. Please follow the steps below to search for specific site information and setup T&C in Laegeren.

## 3. Set up the model for Laegeren

* Open `prepare_data_ch_lae.jl` and provide the required soil texture information for Laegeren. To do this, replace the `X.XX` in the `data["soil"]` dictionary by actual soil texture information. You can consider the soil type to be a sandy loam and refer to the soil texture triangle to obtain reasonable textural values.
* Vegetation cover in Laegeren is simplified to two types, beech and fir, as defined by `Ccrown = [0.55, 0.45]`. When more than one vegetation type is represented, the corresponding parameter arrays contain one element for each type in the same order. For example, `"Knit" => [0.35, 0.20]` assigns a canopy nitrogen decay coefficient of 0.35 to beech and 0.20 to fir. The structure of `vegetationdynamics["high"]` is different: instead of storing two values in each parameter array, it contains two `Dict{String,Any}` objects. The first dictionary contains the parameters for beech, and the second dictionary contains those for fir. Therefore, for `aSE`, `Tcold`, and `Tlo`, edit the values in the first dictionary only. In `prepare_data_ch_lae.jl`, the values labelled `b`–`h` for deciduous beech are intentionally incomplete. Replace these placeholders using Table 1. The second values or dictionaries, corresponding to fir, are already complete and should not be changed. Only the parameters listed in Table 1 need to be filled.

<center>

| Parameter/Location  | Chamau  | BAY   | Davos | Hainich | HARVARD | Laegeren  |
| :-                  | :-      | :-    | :-    | :-      | :-      | :-        |
| Species             | C3 Grassland <br> Managed and smooth meadow grass | H: Evergreen Forest <br>[Norway Spruce]| H: Evergreen Forest <br>[Norway Spruce] | H: Deciduous forest<br>[Beech, ash and maple]| H: Deciduous Forest<br>[Read oak and red maple]|H: Deciduous Forest<br>[Beech]|
| aSE                 | 2       | 0     | 0     | 1       | 1       |     b      |
| Sp_LAI_In           | 0.2     | 0.1   | 0.1   | 0.15    | 0.2     |      c     |
| d_leaf              | 0.8     | 0.25  | 0.25  | 4       | 4       |      d     |
| Ha                  | 55      | 72    | 72    | 76      | 76      |       e   |
| Tcold               | -2      | -20   | -20   | 3       | 5.5     |        f   |
| Tlo                 | 0       | 8.5   | 4.5   | 5       | 5.5     |        g   |
| Vmax                | 96      | 40    | 44    | 76      | 62      |        h   |

*Table 1: Look-up table for some vegetation parameters*

</center>



After the missing parameters are set in Laegeren, the model parameter file is ready. Similar steps can be performed to implement T&C in a new case study. Note, however, that while parameters are provided in Table 1 for simplicity, they would generally be obtained from existing databases—such as SoilGrids and the TRY Plant Trait Database—or from the literature. Automated tools are available to help set up T&C for a new case study; however, using them requires some familiarity with MATLAB. If you need to establish a new case study, for example in your subsequent group project, please contact a teaching assistant for guidance.

## 4. Ready to run the simulation

### 4.1 Prepare the model parameter file

After completing the missing parameters `b`–`h` for deciduous beech, generate the CH-Lae YAML configuration file. For example:

In [ ]:
# Load the general data utilities and the CH-Lae preparation functions.
include(joinpath(@__DIR__, "..", "utils", "data_utils.jl"))
include(joinpath(@__DIR__, "prepare_data_ch_lae.jl")) #include(joinpath(@__DIR__, "prepare_data_ch_lae_solution.jl"))

# # After replacing placeholders X.XX and b–h, create the CH-Lae parameter dictionary.
params = prepare_parameters()

# # Save the model parameters as the YAML configuration file used by T&C.
save_parameters(params, yaml_path)
println("Saved CH-Lae model parameters to: ", yaml_path)

### 4.2 Initialize and run the CH-Lae model

The model requires both the YAML parameter file and the NetCDF forcing file. First, create the CH-Lae NetCDF file from the provided pre-processed meteorological and atmospheric CO₂ input data:

In [ ]:
prepare_netcdf(forcing_input, ca_path, netcdf_path)
println("Saved CH-Lae forcing data to: ", netcdf_path)

Initialize the model using the CH-Lae forcing and parameter files:

In [ ]:
using TethysChloris

FT = Float64
model = initialize_model(FT, netcdf_path, yaml_path)

Use the same model options introduced in Exercise 1. These options enable the soil-temperature, frozen-soil, and vegetation–snow interaction components and specify stable numerical solution methods:

In [ ]:
import Roots

options = ModelOptions(
    SoilTemperature = true,
    FreezingSoil = true,
    VegetationSnowInteractions = true,
    OPT_CR = RootsNonBracketingStrategy(FT, Roots.Order0()),
    OPT_ST = RootsBracketingStrategy(FT, Roots.Brent()),
    OPT_ST2 = RootsBracketingStrategy(FT, Roots.Brent()),
    OPT_VD = ODEOptions(; abstol = 0.05),
)

Finally, run the simulation for the complete 96,432-hour CH-Lae forcing period:

In [ ]:
NN = 96432
diagnostics = run_simulation(model; NN = NN, options)

### 4.3 Save the original simulation results

The YAML file stores model parameters, but the simulated time series remain in memory unless they are saved explicitly. TethysChloris provides the `save` function for this purpose. It creates a results archive containing `results.nc` together with the YAML and NetCDF input files used for that simulation. Here, only the variables required in Part B are included in `results.nc`.

In [ ]:
results_dir = joinpath(@__DIR__, "results")
mkpath(results_dir)

# GPP is reconstructed from its NPP and autotrophic-respiration components.
outputs_to_save = ["NPP_H", "RA_H", "NPP_L", "RA_L", "QE", "Lk"]

original_results_path = joinpath(results_dir, "CH-Lae_original_results.zip")
isfile(original_results_path) && rm(original_results_path)

save(
    model,
    original_results_path;
    options = options,
    outputs = outputs_to_save,
    compress = true,
)

println("Saved original simulation results to: ", original_results_path)

## 5. Result checking and model validation

Model validation compares simulated variables with independent observations to assess how well the selected parameters reproduce the behaviour of the ecosystem. In this exercise, observations from the Swiss FluxNet towers are compared with simulations for Laegeren.

Figures 1 and 2 illustrate two useful forms of comparison: time-series plots reveal temporal patterns and data-quality problems, while scatter plots summarize the agreement between paired observations and simulations. Examine these examples before working through the Laegeren validation workflow below.

<center>
  <figure>
      <img src="assets/images/fig1.webp" alt="Fig. 1">
      <figcaption>Figure 1. An example of results from Fatichi et al. (2014): comparison between the observed (OBS) and simulated (SIM) average daily cycles of net radiation, latent heat, and sensible heat for the location of Chamau. The triangles represent the standard deviations. Scatter plots with the determination coefficients R2 are also shown.</figcaption>
  </figure>
</center>

<center>
  <figure>
      <img src="assets/images/fig2.webp" alt="Fig. 2">
      <figcaption>Figure 2. An example of results from Fatichi et al. (2014): comparison between the observed (OBS) and simulated (SIM) (a) daily and (c) annual GPP for the location of Chamau. A scatter plot with the determination coefficient R2 is shown for (b) daily GPP.</figcaption>
  </figure>
</center>

Before running the validation example, consider the following questions:

- Which observed variables are used to evaluate the model?
- What is the temporal resolution of the observations and the corresponding model outputs?
- Are there missing values or obvious outliers in the observations? How should these be handled before calculating validation metrics?
- What complementary information is provided by a time-series plot and an observation-versus-simulation scatter plot?

### 5.1 Quick visual check of the simulation

Before comparing the simulation with observations, inspect the model output itself. The plotting functions provided by `TethysChloris.Figures` offer a quick overview of LAI, GPP, NPP, and leaf age. For CH-Lae, crown area 1 represents beech and crown area 2 represents fir.

In [ ]:
using TethysChloris.Figures

n_model_days = size(model.state.vegetation.high.NPP, 1)

# Display both figures explicitly so that both remain visible in the notebook.
display(plot_vegetation_dynamics(model, n_model_days, 1))  # beech
display(plot_vegetation_dynamics(model, n_model_days, 2))  # fir

### 5.2 Laegeren validation example

The following cells reproduce the complete Laegeren validation workflow in Julia. They compare latent heat, sensible heat, net radiation, and GPP with the corresponding Swiss FluxNet observations. The observation files for both case-study sites are provided in the `Files_validation` folder and are accessed using paths relative to this notebook.

In [ ]:
using MAT: matread
using Dates: Date
using Statistics: mean, std, cor
using Plots

# Complete observation files are provided for both case-study sites.
# Here, Laegeren is used as the model-validation example.
# Build the path relative to this notebook so that it works on every student's computer.
validation_data_dir = joinpath(@__DIR__, "Files_validation")
laegeren_observation_path = joinpath(validation_data_dir, "Res_CH-Lae.mat")
# chamau_observation_path = joinpath(validation_data_dir, "Res_CH-Cha.mat")

observations = matread(laegeren_observation_path)

The observation record extends beyond the CH-Lae simulation period. Select only observations between the first and last model timestamps, extract the four validation variables, and inspect their data availability. `GPP1` is used as the observation-based GPP estimate for this exercise.

In [ ]:
observation_dates_all = datevec(vec(observations["Date"]))
model_dates = model.forcing.Datam[1:NN]

period_mask =
    (observation_dates_all .>= first(model_dates)) .&
    (observation_dates_all .<= last(model_dates))

@assert count(period_mask) == NN

observation_dates = observation_dates_all[period_mask]
LE_obs = vec(observations["LE"])[period_mask]
H_obs = vec(observations["H"])[period_mask]
Rn_obs = vec(observations["Rn"])[period_mask]
GPP_obs_hourly = vec(observations["GPP1"])[period_mask]

println("Validation period: ", first(observation_dates), " to ", last(observation_dates))
println("Finite LE observations: ", count(isfinite, LE_obs), " / ", NN)
println("Finite H observations:  ", count(isfinite, H_obs), " / ", NN)
println("Finite Rn observations: ", count(isfinite, Rn_obs), " / ", NN)
println("Finite GPP observations: ", count(isfinite, GPP_obs_hourly), " / ", NN)

Latent heat, sensible heat, and net radiation are stored hourly in `model.auxiliary.hydrologic`. GPP is stored daily and is calculated as `NPP + RA`. The high- and low-vegetation contributions are combined and weighted by their fractional crown cover (`Ccrown`).

Hourly observed GPP in µmol CO₂ m⁻² s⁻¹ is averaged by day and converted to gC m⁻² day⁻¹. A day containing a missing hourly value remains missing, rather than being silently filled.

In [ ]:
# Hourly simulated energy fluxes [W m⁻²]
hydro = model.auxiliary.hydrologic
LE_sim = hydro.QE[1:NN]
H_sim = hydro.H[1:NN]
Rn_sim = hydro.Rn[1:NN]

function daily_mean_preserve_missing(values; hours_per_day = 24)
    @assert length(values) % hours_per_day == 0
    return vec(mean(reshape(values, hours_per_day, :); dims = 1))
end

# Convert observed GPP to daily gC m⁻² day⁻¹.
GPP_obs_daily =
    daily_mean_preserve_missing(GPP_obs_hourly) .*
    (12e-6 * 24 * 3600)

# Calculate crown-area-weighted simulated daily GPP.
Ccrown = model.parameters.landcover.Ccrown
GPP_sim_with_initial =
    (
        model.state.vegetation.high.NPP .+
        model.auxiliary.vegetation.high.RA .+
        model.state.vegetation.low.NPP .+
        model.auxiliary.vegetation.low.RA
    ) * Ccrown

# Row 1 is the initial model state; rows 2:end correspond to simulated days.
n_validation_days = length(GPP_obs_daily)
GPP_sim_daily = GPP_sim_with_initial[2:(n_validation_days + 1)]
daily_dates = Date.(model_dates[1:24:end])

### 5.3 Validation metrics

Only timestamps with finite observations and simulations are paired for the calculations below. Three complementary metrics are reported:

- **RMSE** measures the typical magnitude of the simulation error; lower values are better.
- **NSE** compares the simulation with using the observed mean as a predictor; 1 is ideal, 0 indicates equal performance, and negative values indicate poorer performance.
- **R²** measures the strength of the linear association between observations and simulations; values closer to 1 indicate stronger agreement in variation. For this exercise, an R² of approximately 0.4 can already be considered satisfactory for daily GPP validation; the results do not need to match the quality of the published examples above.

In [ ]:
function validation_metrics(observed, simulated)
    valid = isfinite.(observed) .& isfinite.(simulated)
    obs = observed[valid]
    sim = simulated[valid]
    n = length(obs)

    if n < 2
        return (; n, RMSE = NaN, NSE = NaN, R2 = NaN)
    end

    rmse = sqrt(mean((sim .- obs) .^ 2))
    denominator = sum((obs .- mean(obs)) .^ 2)
    nse = iszero(denominator) ? NaN :
          1 - sum((sim .- obs) .^ 2) / denominator
    r2 = iszero(std(obs)) || iszero(std(sim)) ? NaN : cor(obs, sim)^2

    return (; n, RMSE = rmse, NSE = nse, R2 = r2)
end

validation_results = [
    ("Latent heat", validation_metrics(LE_obs, LE_sim)),
    ("Sensible heat", validation_metrics(H_obs, H_sim)),
    ("Net radiation", validation_metrics(Rn_obs, Rn_sim)),
    ("Daily GPP", validation_metrics(GPP_obs_daily, GPP_sim_daily)),
]

for (variable, result) in validation_results
    println(
        variable,
        ": n = ", result.n,
        ", RMSE = ", round(result.RMSE, digits = 3),
        ", NSE = ", round(result.NSE, digits = 3),
        ", R² = ", round(result.R2, digits = 3),
    )
end

### 5.4 Validation plots

For each variable, plot the complete observed and simulated time series together with a scatter plot of all finite paired values. The dashed 1:1 line represents perfect agreement.

In [ ]:
function plot_validation(dates, observed, simulated; variable, unit)
    valid = isfinite.(observed) .& isfinite.(simulated)
    obs = observed[valid]
    sim = simulated[valid]

    time_series = plot(
        dates, observed;
        label = "OBS",
        ylabel = unit,
        title = "$(variable): observed and simulated",
        linewidth = 1,
    )
    plot!(time_series, dates, simulated; label = "SIM", linewidth = 1)

    if length(obs) >= 2
        lower = min(minimum(obs), minimum(sim))
        upper = max(maximum(obs), maximum(sim))
        if lower == upper
            lower -= 1
            upper += 1
        end

        scatter_plot = scatter(
            obs, sim;
            label = false,
            xlabel = "OBS [$(unit)]",
            ylabel = "SIM [$(unit)]",
            title = "$(variable): paired values",
            markersize = 2,
            markerstrokewidth = 0,
            alpha = 0.4,
            xlims = (lower, upper),
            ylims = (lower, upper),
            aspect_ratio = :equal,
        )
        plot!(
            scatter_plot,
            [lower, upper], [lower, upper];
            label = "1:1",
            color = :black,
            linestyle = :dash,
        )
    else
        scatter_plot = plot(
            ;
            xlims = (0, 1),
            ylims = (0, 1),
            legend = false,
            axis = false,
            title = "$(variable): no finite observation–simulation pairs",
        )
    end

    return plot(time_series, scatter_plot; layout = (2, 1), size = (950, 750))
end

In [ ]:
LE_metrics = validation_metrics(LE_obs, LE_sim)
println("Latent heat metrics: ", LE_metrics)
plot_validation(
    observation_dates, LE_obs, LE_sim;
    variable = "Latent heat",
    unit = "W m⁻²",
)

In [ ]:
GPP_metrics = validation_metrics(GPP_obs_daily, GPP_sim_daily)
println("Daily GPP metrics: ", GPP_metrics)
plot_validation(
    daily_dates, GPP_obs_daily, GPP_sim_daily;
    variable = "Daily GPP",
    unit = "gC m⁻² day⁻¹",
)

In [ ]:
H_metrics = validation_metrics(H_obs, H_sim)
println("Sensible heat metrics: ", H_metrics)
plot_validation(
    observation_dates, H_obs, H_sim;
    variable = "Sensible heat",
    unit = "W m⁻²",
)

In [ ]:
Rn_metrics = validation_metrics(Rn_obs, Rn_sim)

if Rn_metrics.n == 0
    println(
        "Net-radiation validation is skipped because no finite Rn observations ",
        "are available for Laegeren during the simulation period.",
    )
else
    println("Net radiation metrics: ", Rn_metrics)
    plot_validation(
        observation_dates, Rn_obs, Rn_sim;
        variable = "Net radiation",
        unit = "W m⁻²",
    )
end

### 5.5 Interpreting the results and improving the parameters

* Focus on identifying discrepancies and calibrating the parameters to improve the fit iteratively. For this exercise, use GPP as the primary calibration target and look for parameter values that improve the validation metrics and temporal pattern. In the Julia configuration, high- and low-vegetation parameters are stored in the `high` and `low` dictionaries. When a vegetation group contains more than one crown area, its parameter arrays follow the beech–fir order described in Section 3.
* Table 2 shows some potentially important parameters that influence the model output. Manually adjust these parameters and re-run the simulation to assess their impact. Try to get the combination of parameters that allows you to best match the data (model calibration) in Laegeren. Note that several techniques exist for automatic model calibration, but this will not be explored here for the sake of time. Also, for complex models such as T&C the setup of an automatic calibration procedure may be quite complicated (given the very large parameter space).
* After adjusting the parameters, observe the changes in GPP and document the results in Table 2. For example, did the variation of parameters increase or decrease GPP? Did they significantly affect the temporal patterns of GPP? In the table, summarize the overall relationship between parameter changes and simulation outcomes, focusing on trends rather than specific statistical metrics. Where possible, provide a physical explanation for the observed behavior, describing why a particular parameter change impacts GPP in the way it does.

For each calibration trial, start from a fresh parameter dictionary and modify the desired entries directly. For high vegetation, index `1` represents deciduous beech and index `2` represents fir. Change one parameter at a time so that its effect can be interpreted clearly.

In [ ]:
# Start each trial from the unchanged default parameters.
trial_params = prepare_parameters()

# Modify the deciduous-beech parameters (index 1).
trial_params["vegetationdynamics"]["high"][1]["Tcold"] = 4.5
# trial_params["vegetation"]["high"]["Vmax"][1] = 80.0
# trial_params["vegetationdynamics"]["high"][1]["Tlo"] = 4.0

# Save this parameter set separately from the original configuration.
trial_yaml_path = joinpath(data_dir, "CH-Lae_trial.yaml")
save_parameters(trial_params, trial_yaml_path)

# Run the trial, then re-run the validation cells above to inspect the response.
model = initialize_model(FT, netcdf_path, trial_yaml_path)
diagnostics = run_simulation(model; NN = NN, options)

# Save results.nc and the trial input files in a TethysChloris results archive.
trial_results_path = joinpath(results_dir, "CH-Lae_calibration_trial_results.zip")
isfile(trial_results_path) && rm(trial_results_path)
save(
    model,
    trial_results_path;
    options = options,
    outputs = outputs_to_save,
    compress = true,
)
println("Saved calibration-trial results to: ", trial_results_path)

<center>

| Parameters        | Unit                    | Meaning                                                               | Impact on GPP/LAI pattern |
| :-:               | :-:                     | :-:                                                                   | :-:                       |
| Tcold             | °C                      | Air-temperature threshold for leaf shedding                            | &nbsp;                    |
| Vmax              | μmol CO2/m<sup>2</sup>s | Maximum Rubisco capacity at 25°C at leaf level                         | &nbsp;                    |
| Tlo               | °C                      | Temperature threshold for leaf onset                                   | &nbsp;                    |

Table 2. Effect of variations in some key parameters on daily GPP/LAI dynamics

</center>

# Part B: Sensitivity analysis and scenarios

In this second part, we will perform a sensitivity analysis and investigate simple climate change scenarios.

## 1. Sensitivity analysis

Modify one vegetation or root-depth property at a time and quantify its effect on GPP, latent heat (`QE`), and leakage. In the example below, change the value of `scenario` to select an experiment. Each experiment starts from the original YAML and NetCDF inputs, runs independently, and saves its selected output variables to the `results` folder.

In [ ]:
# Select ONE sensitivity experiment.
scenario = :root_depth_500

# Other choices:
# scenario = :root_depth_1200
# scenario = :specific_leaf_area_minus_50
# scenario = :specific_leaf_area_plus_50
# scenario = :vmax_50

scenario_name = String(scenario)
scenario_params = prepare_parameters()
scenario_yaml_path = joinpath(data_dir, "CH-Lae_$(scenario_name).yaml")
scenario_netcdf_path = joinpath(data_dir, "CH-Lae_$(scenario_name).nc")

# Start from an independent copy of the original forcing and initial conditions.
cp(netcdf_path, scenario_netcdf_path; force = true)

if scenario == :root_depth_500
    NCDataset(scenario_netcdf_path, "a") do ds
        ds.group["high_vegetation"]["ZR95"][:] = [500.0, 500.0]
    end
elseif scenario == :root_depth_1200
    NCDataset(scenario_netcdf_path, "a") do ds
        ds.group["high_vegetation"]["ZR95"][:] = [1200.0, 1200.0]
    end
elseif scenario == :specific_leaf_area_minus_50
    scenario_params["vegetation"]["high"]["Sl"] .*= 0.5
    for crown in eachindex(scenario_params["vegetationdynamics"]["high"])
        scenario_params["vegetationdynamics"]["high"][crown]["Sl"] *= 0.5
    end
elseif scenario == :specific_leaf_area_plus_50
    scenario_params["vegetation"]["high"]["Sl"] .*= 1.5
    for crown in eachindex(scenario_params["vegetationdynamics"]["high"])
        scenario_params["vegetationdynamics"]["high"][crown]["Sl"] *= 1.5
    end
elseif scenario == :vmax_50
    # Index 1 is deciduous beech; fir already has Vmax = 50.
    scenario_params["vegetation"]["high"]["Vmax"][1] = 50.0
else
    error("Unknown sensitivity scenario: $scenario")
end

save_parameters(scenario_params, scenario_yaml_path)

scenario_model = initialize_model(FT, scenario_netcdf_path, scenario_yaml_path)
scenario_diagnostics = run_simulation(scenario_model; NN = NN, options)

scenario_results_path = joinpath(results_dir, "CH-Lae_$(scenario_name)_results.zip")
isfile(scenario_results_path) && rm(scenario_results_path)
save(
    scenario_model,
    scenario_results_path;
    options = options,
    outputs = outputs_to_save,
    compress = true,
)
println("Saved scenario results to: ", scenario_results_path)

<center>

| Variables                           | GPP     | ET (Latent heat, QE)  | Leakage | 
| :----------------------------------:| :---:   | :-------------------: | :-----: |
| Root depth decrease to 500mm        | -   | -                | +    |
| Root depth increase to 1200mm       | &nbsp;  | &nbsp;                | &nbsp;  |
| Specific leaf area -50%             | &nbsp;  | &nbsp;                | &nbsp;  |
| Specific leaf area +50%             | &nbsp;  | &nbsp;                | &nbsp;  |
| Vmax = 50 [μmol CO2/m<sup>2</sup>s] | &nbsp;  | &nbsp;                | &nbsp;  |

</center>

## 2. Analysis of simple scenarios

Run additional simulations with modified meteorological or atmospheric forcing and quantify the changes in the target ecohydrological variables. Change the value of `scenario` below to select one experiment. The original NetCDF file is copied before modification, so the baseline forcing remains unchanged.

In [ ]:
# Select ONE forcing scenario.
scenario = :temperature_plus_3C

# Other choices:
# scenario = :precipitation_minus_50
# scenario = :co2_plus_300

scenario_name = String(scenario)
scenario_params = prepare_parameters()
scenario_yaml_path = joinpath(data_dir, "CH-Lae_$(scenario_name).yaml")
scenario_netcdf_path = joinpath(data_dir, "CH-Lae_$(scenario_name).nc")

# Start from the original parameters and an independent copy of the forcing file.
save_parameters(scenario_params, scenario_yaml_path)
cp(netcdf_path, scenario_netcdf_path; force = true)

NCDataset(scenario_netcdf_path, "a") do ds
    if scenario == :temperature_plus_3C
        ds["Ta"][:] = ds["Ta"][:] .+ 3.0
    elseif scenario == :precipitation_minus_50
        ds["Pr"][:] = ds["Pr"][:] .* 0.5
    elseif scenario == :co2_plus_300
        ds["Ca"][:] = ds["Ca"][:] .+ 300.0
    else
        error("Unknown forcing scenario: $scenario")
    end
end

scenario_model = initialize_model(FT, scenario_netcdf_path, scenario_yaml_path)
scenario_diagnostics = run_simulation(scenario_model; NN = NN, options)

scenario_results_path = joinpath(results_dir, "CH-Lae_$(scenario_name)_results.zip")
isfile(scenario_results_path) && rm(scenario_results_path)
save(
    scenario_model,
    scenario_results_path;
    options = options,
    outputs = outputs_to_save,
    compress = true,
)
println("Saved scenario results to: ", scenario_results_path)

<center>

| Variables               | GPP     | ET (Latent heat, QE)  | Leakage | 
| :----------------------:| :---:   | :-------------------: | :-----: |
| Ta +3°C                 | &nbsp;  | &nbsp;                | &nbsp;  |
| Pr -50%                 | &nbsp;  | &nbsp;                | &nbsp;  |
| CO<sub>2</sub> +300ppm  | &nbsp;  | &nbsp;                | &nbsp;  |

</center>

## 3. Load and compare saved results

Each archive produced by `save` contains a `results.nc` file. The code below extracts that NetCDF file temporarily, reads the saved variables with `NCDataset`, reconstructs total GPP, and compares a selected scenario with the original simulation. Cumulative GPP and leakage are compared using sums. Because `QE` is an energy flux in W m⁻², its temporal mean is used. Positive percentages indicate an increase relative to the original simulation, whereas negative percentages indicate a decrease.

In [ ]:
using NCDatasets: NCDataset
using YAML: load_file
using Statistics: mean

# ZipFile is already used internally by TethysChloris.save.
ZipArchive = TethysChloris.IO.ZipFile

function load_saved_outputs(archive_path)
    mktempdir() do temporary_dir
        results_nc_path = joinpath(temporary_dir, "results.nc")
        parameters_path = joinpath(temporary_dir, "parameters.yaml")

        reader = ZipArchive.Reader(archive_path)
        try
            results_entry = nothing
            parameters_entry = nothing

            for file in reader.files
                file.name == "results.nc" && (results_entry = file)
                endswith(lowercase(file.name), ".yaml") && (parameters_entry = file)
            end

            results_entry === nothing &&
                error("Could not find results.nc inside $archive_path")
            parameters_entry === nothing &&
                error("Could not find the YAML parameter file inside $archive_path")

            open(results_nc_path, "w") do io
                write(io, read(results_entry))
            end
            open(parameters_path, "w") do io
                write(io, read(parameters_entry))
            end
        finally
            close(reader)
        end

        parameters = load_file(parameters_path)
        Ccrown = Float64.(parameters["landcover"]["Ccrown"])

        return NCDataset(results_nc_path, "r") do ds
            GPP_with_initial =
                (
                    Array(ds["NPP_H"][:, :]) .+
                    Array(ds["RA_H"][:, :]) .+
                    Array(ds["NPP_L"][:, :]) .+
                    Array(ds["RA_L"][:, :])
                ) * Ccrown

            (
                GPP_daily = collect(GPP_with_initial[2:end]),
                QE_hourly = vec(Array(ds["QE"][:])),
                Lk_hourly = vec(Array(ds["Lk"][:])),
            )
        end
    end
end

# Enter the name of any scenario that has already been run and saved.
scenario_name = "root_depth_500"
# Examples: "calibration_trial", "specific_leaf_area_minus_50",
# "temperature_plus_3C", "precipitation_minus_50", or "co2_plus_300".

original_results_path =
    joinpath(@__DIR__, "results", "CH-Lae_original_results.zip")
scenario_results_path =
    joinpath(@__DIR__, "results", "CH-Lae_$(scenario_name)_results.zip")

original_results = load_saved_outputs(original_results_path)
scenario_results = load_saved_outputs(scenario_results_path)

GPP_original = sum(original_results.GPP_daily)
GPP_scenario = sum(scenario_results.GPP_daily)

QE_original = mean(original_results.QE_hourly)
QE_scenario = mean(scenario_results.QE_hourly)

Lk_original = sum(original_results.Lk_hourly)
Lk_scenario = sum(scenario_results.Lk_hourly)

percent_difference(scenario_value, original_value) =
    100 * (scenario_value / original_value - 1)

println("Scenario: ", scenario_name)
println(
    "GPP difference = ",
    round(percent_difference(GPP_scenario, GPP_original), digits = 2),
    " %",
)
println(
    "QE difference  = ",
    round(percent_difference(QE_scenario, QE_original), digits = 2),
    " %",
)
println(
    "Lk difference  = ",
    round(percent_difference(Lk_scenario, Lk_original), digits = 2),
    " %",
)

### Find out more

[Laboratory of Catchment Hydrology and Geomorphology](https://www.epfl.ch/labs/change/)
[T&C source code (MATLAB)](https://github.com/simonefatichi/TeC_Source_Code)


### References 

Fatichi, S., Ivanov, V. Y., & Caporali, E. (2012). A mechanistic ecohydrological model to investigate complex interactions in cold and warm water‐controlled environments: 1. Theoretical framework and plot‐scale analysis. *Journal of Advances in Modeling Earth Systems, 4*(2).
Fatichi, S., Zeeman, M. J., Fuhrer, J., & Burlando, P. (2014). Ecohydrological effects of management on subalpine grasslands: From local to catchment scale. *Water Resources Research, 50*(1), 148-164